# 07 – Silver MERGE edge case tests

Shows that the batch log and quarantine work correctly in edge cases.

1. Silver is reset, then batches 1 → 3 are applied with the runner.

2. `batch_0002` is run a second time: the log should mark all 4 tables SKIPPED, and silver should not change.

3. The edge case IDs are checked in silver and quarantine.

This cell imports the `src/silver_merge.py` module.

In [0]:
import os
import sys
import importlib

from pyspark.sql import functions as F

# Notebook is under notebooks/; the src folder is at the repo root
repo_root = os.path.dirname(os.getcwd())
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

import silver_merge
importlib.reload(silver_merge)

## 1. Reset

The 4 silver tables return to the bronze snapshot (`_change_ts` = 1998-12-31), CDF is enabled again on `card`, and `batch_log` and `quarantine` are emptied.

Expected output: row counts equal bronze (client 5,369, account 4,500, card 892, loan 682); `batch_log` and `quarantine` 0.

In [0]:
silver_merge.reset_silver(spark)

# Start state after reset: silver = bronze, log and quarantine empty
for table_name in ["client", "account", "card", "loan"]:
    row_count = spark.table(f"workspace.silver.{table_name}").count()
    print(table_name, row_count)

print("batch_log", spark.table("workspace.silver.batch_log").count())
print("quarantine", spark.table("workspace.silver.quarantine").count())

## 2. Batch 1 → 3 (runner)

In each batch, the 4 tables are processed with `run_batch`: log check, missing keys to quarantine, MERGE, log SUCCESS.

Expected output: 12 rows in `batch_log`, all SUCCESS. upd / del / ins counts are the same as in 06_silver_merge. `rows_skipped` only in batch 3: card 1 (101254) and loan 1 (107308).

In [0]:
CDC_TABLES = ["client", "account", "card", "loan"]

# Order matters: each batch builds on the previous one
for batch_id in [1, 2, 3]:
    for table_name in CDC_TABLES:
        silver_merge.run_batch(spark, table_name, batch_id)

df_log = spark.table("workspace.silver.batch_log")
display(
    df_log.select("table_name", "batch_id", "status", "rows_read",
                  "rows_updated", "rows_deleted", "rows_inserted", "rows_skipped")
          .orderBy("batch_id", "table_name")
)

## 3. State before the rerun

The row count and the last MERGE version of each silver table are saved. The same values are expected after the `batch_0002` rerun.

Expected output: client 5,383, account 4,500, card 890, loan 682; `batch_log` 12, `quarantine` 2.

In [0]:
def take_snapshot():
    snapshot = {}
    for table_name in CDC_TABLES:
        full_name = f"workspace.silver.{table_name}"
        row_count = spark.table(full_name).count()
        # A MERGE creates a new version; automatic operations such as OPTIMIZE are not counted
        df_history = spark.sql(f"DESCRIBE HISTORY {full_name}")
        last_merge_version = df_history.filter(F.col("operation") == "MERGE").agg(F.max("version")).collect()[0][0]
        snapshot[table_name] = {"rows": row_count, "last_merge_version": last_merge_version}
    return snapshot

snapshot_before = take_snapshot()
for table_name in CDC_TABLES:
    print(table_name, snapshot_before[table_name])

log_count_before = spark.table("workspace.silver.batch_log").count()
quarantine_count_before = spark.table("workspace.silver.quarantine").count()
print("batch_log", log_count_before, "| quarantine", quarantine_count_before)

## 4. Rerun of the same batch (`batch_0002`)

Batch 2 is passed to `run_batch` a second time for the four tables. It is already SUCCESS in the log, so it should not reach MERGE.

Expected output: `run_batch` returns SKIPPED for each table and `unchanged` is True; `batch_log` 12 → 12, `quarantine` 2 → 2.

In [0]:
# The same batch arrives a second time; tests the log protection
run_results = {}
for table_name in CDC_TABLES:
    result = silver_merge.run_batch(spark, table_name, 2)
    run_results[table_name] = result

snapshot_after = take_snapshot()

# Row count and last MERGE version must be the same before and after
for table_name in CDC_TABLES:
    before = snapshot_before[table_name]
    after = snapshot_after[table_name]
    unchanged = before == after
    print(table_name, "| run_batch:", run_results[table_name], "| before:", before, "| after:", after, "| unchanged:", unchanged)

log_count_after = spark.table("workspace.silver.batch_log").count()
quarantine_count_after = spark.table("workspace.silver.quarantine").count()
print("batch_log", log_count_before, "->", log_count_after)
print("quarantine", quarantine_count_before, "->", quarantine_count_after)

## 5. Edge case checks

| Edge case | ID | Expected |
|---|---|---|
| Same key twice in one batch (U → U) | client 435, 4206 | Latest `change_ts` wins: district 28 and 27 |
| New card upgraded in the same batch (I → U) | card 1253 | Inserted with its latest values: `gold` |
| D / U for a missing key | card 101254, loan 107308 | Not in silver, in quarantine |

Expected output: `ok` = true in all rows of `df_checks`, followed by the full quarantine table (2 rows).

In [0]:
checks = []

# U → U: the latest change_ts must win in dedup
client_expected = [(435, 28), (4206, 27)]
for client_id, expected_district in client_expected:
    row = spark.table("workspace.silver.client").filter(F.col("client_id") == client_id).collect()[0]
    checks.append((f"client {client_id} district_id", str(expected_district), str(row["district_id"])))

# I → U: the card was not in the target; it must be inserted with its latest values (gold)
row = spark.table("workspace.silver.card").filter(F.col("card_id") == 1253).collect()[0]
checks.append(("card 1253 type", "gold", row["type"]))

# Missing key: must not enter silver; must have one row in quarantine
df_quarantine = spark.table("workspace.silver.quarantine")
missing_keys = [("card", "card_id", 101254), ("loan", "loan_id", 107308)]
for table_name, key_column, key_value in missing_keys:
    silver_count = spark.table(f"workspace.silver.{table_name}").filter(F.col(key_column) == key_value).count()
    checks.append((f"{table_name} {key_value} in silver", "0", str(silver_count)))

    quarantine_filter = (F.col("table_name") == table_name) & (F.col("key_value") == str(key_value))
    quarantine_count = df_quarantine.filter(quarantine_filter).count()
    checks.append((f"{table_name} {key_value} in quarantine", "1", str(quarantine_count)))

df_checks = spark.createDataFrame(checks, ["check", "expected", "actual"])
df_checks = df_checks.withColumn("ok", F.col("expected") == F.col("actual"))
display(df_checks)

display(df_quarantine)